In [ ]:
spark.conf.set(
  "fs.azure.account.key.retailprojectstorageacct.dfs.core.windows.net",
  "Aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaasssssssssssddsdsds=="
)

In [ ]:
dbutils.fs.ls("abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze")

[FileInfo(path='abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/customer/', name='customer/', size=0, modificationTime=1767307453000),
 FileInfo(path='abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/product/', name='product/', size=0, modificationTime=1767307419000),
 FileInfo(path='abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/store/', name='store/', size=0, modificationTime=1767307430000),
 FileInfo(path='abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/transaction/', name='transaction/', size=0, modificationTime=1767307441000)]

In [ ]:
dbutils.fs.ls("abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/transaction/")

[FileInfo(path='abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/transaction/dbo.transactions.parquet', name='dbo.transactions.parquet', size=1757, modificationTime=1767318646000)]

In [ ]:
# Read raw data from Bronze layer
df_transactions = spark.read.parquet("abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/transaction/")
df_store = spark.read.parquet("abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/store/")
df_product = spark.read.parquet("abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/product/")
df_customers = spark.read.parquet("abfss://retail@retailprojectstorageacct.dfs.core.windows.net/bronze/customer/")
display(df_transactions)

transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01
4,120,8,1,1,2024-11-02
5,105,5,2,1,2025-03-17
6,110,7,3,5,2025-01-04
7,110,7,2,5,2025-01-01
8,126,7,5,2,2025-06-08
9,123,1,3,2,2024-10-08
10,124,2,2,5,2024-08-27


In [ ]:
%sql
-- Transformation for Transactions (Silver Layer). Convert types and clean data
CREATE OR REPLACE TABLE cleaned_transactions AS
SELECT 
    CAST(transaction_id AS INT) AS transaction_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(product_id AS INT) AS product_id,
    CAST(store_id AS INT) AS store_id,
    CAST(quantity AS INT) AS quantity,
    CAST(transaction_date AS DATE) AS transaction_date
FROM df_transactions;

-- Transformation for Products (Silver Layer)
CREATE OR REPLACE TABLE cleaned_products AS
SELECT 
    CAST(product_id AS INT) AS product_id,
    product_name,
    category,
    CAST(price AS DOUBLE) AS price
FROM df_product;

-- Transformation for Stores (Silver Layer)
CREATE OR REPLACE TABLE cleaned_stores AS
SELECT 
    CAST(store_id AS INT) AS store_id,
    store_name,
    location
FROM df_store;

-- Transformation for Customers with De-duplication (Silver Layer)
--  Clean Customers Table (Remove duplicates)
CREATE OR REPLACE TABLE cleaned_customers AS
        SELECT DISTINCT
        customer_id, 
        first_name, 
        last_name, 
        email, 
        city, 
        registration_date
    FROM df_customers
)


In [ ]:
%sql
-- Join all table together to Create one Silver (Joined) table
CREATE OR REPLACE TABLE retail_silver_cleaned AS
SELECT 
    t.*,                            -- All columns from cleaned_transactions
    c.*,                            -- Specific columns from cleaned_customers
    p.*,                            -- Specific columns from cleaned_products 
    s.*,                            -- Specific columns from cleaned_stores
    (t.quantity * p.price) AS total_amount
FROM cleaned_transactions t
INNER JOIN cleaned_customers c
    ON t.customer_id = c.customer_id
INNER JOIN cleaned_products p
    ON t.product_id = p.product_id
INNER JOIN dcleaned_stores s
    ON t.store_id = s.store_id;

In [ ]:
SELECT * FROM retail_silver_cleaned

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,city,registration_date,product_name,category,price,store_name,location,total_amount
5,7,101,28,3,2024-11-15,Ravi,Yadav,user101@example.com,Delhi,2023-09-14,Smartwatch,Electronics,4999.0,Mega Plaza,Chennai,14997.0
3,1,102,11,2,2024-08-11,Nina,Joshi,user102@example.com,Mumbai,2024-01-21,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,1599.98
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
3,1,105,21,5,2024-10-02,Riya,Singh,user105@example.com,Chennai,2023-06-28,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,3999.95
2,5,105,5,1,2025-03-17,Riya,Singh,user105@example.com,Chennai,2023-06-28,Notebook Set,Stationery,149.0,High Street Store,Delhi,149.0
4,3,105,2,5,2024-11-12,Riya,Singh,user105@example.com,Chennai,2023-06-28,Yoga Mat,Fitness,499.0,Downtown Mini Store,Pune,2495.0
3,9,107,22,4,2024-11-16,Priya,Kapoor,user107@example.com,Ahmedabad,2023-05-12,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
1,5,108,12,4,2025-05-26,Rahul,Verma,user108@example.com,Kolkata,2023-08-19,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
5,8,109,17,5,2024-07-10,Pooja,Mehta,user109@example.com,Delhi,2024-04-01,Desk Organizer,Accessories,399.0,Mega Plaza,Chennai,1995.0


In [ ]:
%sql
--Create External Location
DROP EXTERNAL LOCATION IF EXISTS retail_silver_loc;

CREATE EXTERNAL LOCATION retail_silver_loc
URL 'abfss://retail@retailprojectstorageacct.dfs.core.windows.net/retail/silver'
WITH (STORAGE CREDENTIAL retailproject)
COMMENT 'External location for silver layer';


In [ ]:
%sql
-- Grant Permissions
GRANT READ FILES, WRITE FILES
ON EXTERNAL LOCATION retail_silver_loc
TO `account users`;


In [ ]:
%sql
--# dump the clean data to adls silver layer location
CREATE OR REPLACE TABLE retail_silver_cleaned
USING DELTA
LOCATION 'abfss://retail@retailprojectstorageacct.dfs.core.windows.net/silver/'
AS
SELECT * FROM retail_silver_cleaned;

In [ ]:
-- Load data from silver layer to perform calculations.
SELECT *
FROM delta.`abfss://retail@retailprojectstorageacct.dfs.core.windows.net/silver/`;

In [ ]:
%sql
SELECT * FROM retail_silver_cleaned

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,city,registration_date,product_name,category,price,store_name,location,total_amount
5,7,101,28,3,2024-11-15,Ravi,Yadav,user101@example.com,Delhi,2023-09-14,Smartwatch,Electronics,4999.0,Mega Plaza,Chennai,14997.0
3,1,102,11,2,2024-08-11,Nina,Joshi,user102@example.com,Mumbai,2024-01-21,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,1599.98
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
3,1,105,21,5,2024-10-02,Riya,Singh,user105@example.com,Chennai,2023-06-28,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,3999.95
2,5,105,5,1,2025-03-17,Riya,Singh,user105@example.com,Chennai,2023-06-28,Notebook Set,Stationery,149.0,High Street Store,Delhi,149.0
4,3,105,2,5,2024-11-12,Riya,Singh,user105@example.com,Chennai,2023-06-28,Yoga Mat,Fitness,499.0,Downtown Mini Store,Pune,2495.0
3,9,107,22,4,2024-11-16,Priya,Kapoor,user107@example.com,Ahmedabad,2023-05-12,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
1,5,108,12,4,2025-05-26,Rahul,Verma,user108@example.com,Kolkata,2023-08-19,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
5,8,109,17,5,2024-07-10,Pooja,Mehta,user109@example.com,Delhi,2024-04-01,Desk Organizer,Accessories,399.0,Mega Plaza,Chennai,1995.0


In [ ]:
-- Perform calculation
%sql
CREATE OR REPLACE TABLE gold_retail_summary AS
SELECT 
    transaction_date,
    product_id,
    product_name,
    category,
    store_id,
    store_name,
    location,
    SUM(quantity) AS total_quantity_sold,
    SUM(total_amount) AS total_sales_amount,
    COUNT(DISTINCT transaction_id) AS number_of_transactions,
    AVG(total_amount) AS average_transaction_value
FROM silver_df
GROUP BY 
    transaction_date,
    product_id,
    product_name,
    category,
    store_id,
    store_name,
    location;

In [ ]:

select * from gold_retail_summary

transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2025-05-26,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,4,596.0,1,596.0
2024-07-14,1,Wireless Mouse,Electronics,5,Mega Plaza,Chennai,1,799.99,1,799.99
2024-07-17,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,5,3999.95,1,3999.95
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2025-06-03,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45


In [ ]:
%sql
-- Creating the Gold Table and saving it to the specific ADLS path
CREATE OR REPLACE TABLE retail_gold_sales_final
USING DELTA
LOCATION 'abfss://unity-catalog-storage@dbstoragenwvfhtacdxxl2.dfs.core.windows.net/7405608375731138/retail/gold/'
AS 
SELECT * FROM gold_retail_summary;